In [4]:
import scraping
import create_rawdf
import pandas as pd
from bs4 import BeautifulSoup
import pickle
from tqdm import tqdm
from pathlib import Path
from selenium.webdriver.chrome.options import Options
import time
import re
import chardet
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from urllib.request import Request, urlopen
from bs4 import BeautifulSoup
from tqdm import tqdm
from pathlib import Path
from io import StringIO


DATA_DIR = Path("..", "data")
HTML_DIR = DATA_DIR / "html"
HTML_RACE_DIR = HTML_DIR / "race"
DATA_RAWDF = DATA_DIR / "rawdf"

# 第一回

## 開催日の取得

In [ ]:
import time
from tqdm.notebook import tqdm

def scrape_kaisai_date(from_, to_):
  """
  from_とto_をyyyy-mm(ex:2024-01)の形で設定すると開催日時を取ってくる
  """
  kaisai_date_list = []
  for date in tqdm(pd.date_range(from_, to_, freq="MS")):
    year_ = date.year
    month_ = date.month
    #url = "https://race.netkeiba.com/top/calendar.html?year="+str(year_)+"&month="+str(month_)
    url = f"https://race.netkeiba.com/top/calendar.html?year={year_}&month={month_}"
    #print(url)
    headers = {
      'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'
    }

    #urlから情報を取る。headerないとエラー
    req = Request(url, headers=headers)
    response = urlopen(req).read()
    #今後の処理のためhtmlをデコードする
    encoding = chardet.detect(response)['encoding']
    html = response.decode(encoding)
    time.sleep(1)
    #要素取り出すためにbeautiful soupライブラリを用いる
    soup = BeautifulSoup(html, 'html.parser')
    #Calender_Tablerのなかからa要素を抜く
    a_list = soup.find("table",class_="Calendar_Table").find_all("a")
    for a in a_list:
      kaisai_date = re.findall(r"kaisai_date=(\d{8})", a["href"])[0]
      kaisai_date_list.append(kaisai_date)

  return kaisai_date_list

# 第二回

## race_idの取得

In [ ]:
url = "https://race.netkeiba.com/top/race_list.html?kaisai_date=20241102"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'
}

req = Request(url, headers=headers)
response = urlopen(req).read()
encoding = chardet.detect(response)['encoding']
html = response.decode(encoding)
soup = BeautifulSoup(html)
soup.find("div", class_ = "RaceList_box")

取得できない -> Javaで書かれている場所なのでChromeDriverを使う

In [ ]:
def scrape_race_id_list(kaisai_date_list: list[str]):
    chrome_options = Options()
    chrome_options.add_argument("--headless") # 新しいウィンドウが開かなくなる
    chrome_options.binary_location = "/usr/bin/google-chrome"
    driver_path = ChromeDriverManager().install()
    #print(driver_path)

    driver = webdriver.Chrome(service = Service(driver_path), options = chrome_options)
    race_id_list = []

    with webdriver.Chrome(service = Service(driver_path), options = chrome_options) as driver:
        for kaisai_date in tqdm(kaisai_date_list):
            url = f"https://race.netkeiba.com/top/race_list.html?kaisai_date={kaisai_date}"
            # print(url)

            # タイムアウトを設定（秒単位)
            driver.set_page_load_timeout(60)
            try:
                driver.get(url)  # URLにアクセス
                time.sleep(1)
                li_list = driver.find_elements(By.CLASS_NAME, "RaceList_DataItem")
                for li in li_list:
                    href = li.find_element(By.TAG_NAME, "a").get_attribute("href")
                    race_id = re.findall(r"race_id=(\d{12})", href)[0]
                    race_id_list.append(race_id)
            except:
                print(f"stopped at {url}")
                # print(traceback.format_exc())
                break
        return race_id_list

In [ ]:
# スクリプトのチェック
print(dir(scraping))

In [ ]:
# レース開催日時を取得後、レースIDを取得
kaisai_date_list = scraping.scrape_kaisai_date(from_= "2018-01", to_ = "2024-11")
# kaisai_date_list = scraping.scrape_kaisai_date(from_= "2018-01", to_ = "2024-01")
race_id_list = scraping.scrape_race_id_list(kaisai_date_list)
len(race_id_list)

In [ ]:
# race_id一覧の保存
with open("race_id_list.pickle", "wb") as f:
    pickle.dump(race_id_list, f)

# 第三回

## レースページの取得

In [ ]:
import pickle
with open("race_id_list.pickle", "wb") as f:
    pickle.dump(race_id_list, f)

In [ ]:
import pickle
import scraping

with open("race_id_list.pickle", "rb") as f:
    race_id_list = pickle.load(f)
html_paths_race = scraping.scrape_html_race(race_id_list=race_id_list)

In [ ]:
#5401, 6621, 7607, 8250, 9407, 13969, 21687, 22649, 24999, 
import scraping
import create_rawdf
html_paths_race = list(scraping.HTML_RACE_DIR.glob("*.bin"))
html_paths_race[24999]

PosixPath('../data/html/race/201805010304.bin')

## レース結果のテーブルをすべて結合 ->race.csv

In [ ]:
import scraping
import create_rawdf
html_paths_race = list(scraping.HTML_RACE_DIR.glob("*.bin"))

results = create_rawdf.create_results(html_path_list=html_paths_race)

In [ ]:
results.isnull().sum()

In [ ]:
results.reset_index()[["race_id", "horse_id"]].duplicated().sum()

# 第四回

## race.csvからhorse_idをすべて取得、スクレイピング

In [ ]:
import pandas as pd
from pathlib import Path

path = Path("..", "data", "rawdf", "results.csv")
results = pd.read_csv(path, sep="\t")
horse_id_list = results["horse_id"].unique()
horse_id_list

In [ ]:
import scraping
scraping.scrape_html_horse(horse_id_list=horse_id_list)

## 馬ページのレース結果をすべて取得、結合 ->horse_results.csv

In [2]:
# パスをすべて取得
import scraping
html_paths_horse = list(scraping.HTML_HORSE_JAVA_DIR.glob("*.bin"))
len(html_paths_horse)

35875

In [ ]:
# エラー箇所4294(2017103510.bin),5817(2017102652.bin),5999(2016101224.bin),6438(2015100939.bin),17265(2019103592.bin),20504(2017104786.bin),26177(2018105188.bin),27382(2017103450.bin),
# これらのbinファイルがresults.csvの最後にあるかどうかチェック
html_paths_horse[35874]

PosixPath('../data/html/horse_java/2015103892.bin')

In [10]:
import create_rawdf

# 表の作成
horse_results = create_rawdf.create_horse_results(html_paths_horse[:4652],save_filename="horse_results.csv")
horse_results

100%|██████████| 4652/4652 [40:18<00:00,  1.92it/s]
/home/mech-user/KEI.AI/common/src/create_rawdf.py:245: DtypeWarning: Columns (29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  既存のrawdfに新しいデータを追加して保存する関数


,horse_id,日付,開催,天気,R,レース名,映像,頭数,枠番,馬番,...,着差,ﾀｲﾑ指数,通過,ペース,上り,馬体重,厩舎ｺﾒﾝﾄ,備考,勝ち馬(2着馬),賞金
0,2021100561,2024/12/15,5中山6,晴,10.0,北総S(3勝クラス),NaN,16.0,1.0,2,...,-0.2,**,15-15-13-10,36.8-38.8,37.5,514(+14),NaN,NaN,(マンマリアーレ),1877.1
1,2021100561,2024/09/29,3中京9,曇,10.0,白川郷S(3勝クラス),NaN,15.0,3.0,4,...,0.2,**,8-10-5-4,37.5-36.7,36.5,500(0),NaN,NaN,ディープリボーン,749.4
2,2021100561,2024/09/01,3新潟8,晴,10.0,両津湾特別(2勝クラス),NaN,15.0,6.0,11,...,0.0,**,8-8-7-5,36.2-37.8,37.1,500(+2),NaN,NaN,(ボールドゾーン),1586.4
3,2021100561,2024/08/04,2新潟4,晴,7.0,レパードS(GIII),NaN,15.0,4.0,6,...,0.9,**,11-11-9-8,35.7-37.8,37.7,498(0),NaN,NaN,ミッキーファイト,370.0
4,2021100561,2024/06/05,大井,晴,11.0,東京ダービー競走(JpnI),NaN,16.0,2.0,4,...,2.4,**,10-8-7-7,37.5-37.7,39.1,498(+5),NaN,NaN,ラムジェット,500.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107426,2017105317,2020/05/31,2東京12,曇,11.0,東京優駿(GI),NaN,18.0,1.0,2,...,2.2,**,8-8-9-8,36.8-34.3,36.0,454(-2),NaN,NaN,コントレイル,NaN
107427,2017105317,2020/03/28,2阪神1,曇,11.0,毎日杯(GIII),NaN,10.0,2.0,2,...,0.1,**,7-4,35.4-36.2,35.7,456(-6),NaN,NaN,サトノインプレッサ,1511.0
107428,2017105317,2020/02/09,2京都4,曇,11.0,きさらぎ賞(GIII),NaN,8.0,8.0,8,...,0.1,**,5-5,36.7-34.1,33.7,462(+4),NaN,NaN,コルテジア,954.2
107429,2017105317,2019/11/16,5東京5,晴,11.0,東京スポーツ杯2歳S(GIII),NaN,8.0,5.0,5,...,0.8,**,6-7-7,35.3-33.9,33.6,458(-2),NaN,NaN,コントレイル,1308.6


In [13]:
duplicates = horse_results[horse_results.duplicated(keep=False)]  # keep=Falseで全ての重複行を取得

# 結果を表示
if not duplicates.empty:
    print("重複している行:")
    print(duplicates)
else:
    print("重複している行はありません。")

重複している行はありません。


In [11]:
horse_results.iloc[:, 28:32]

,賞金
0,1877.1
1,749.4
2,1586.4
3,370.0
4,500.0
...,...
107426,NaN
107427,1511.0
107428,954.2
107429,1308.6


# 第六回

## レース情報テーブルの作成 -> race_info.csv

リストから特定の文字列を取り出すときはcreate_race_infoを参照を参照

In [ ]:
DATA_DIR = Path("..", "data")
HTML_DIR = DATA_DIR / "html"
HTML_RACE_DIR = HTML_DIR / "race"
html_path_list = list(HTML_RACE_DIR.glob("*.bin")) # すべてのレースのHTMLを取得
len(html_path_list)

NameError: name 'Path' is not defined

## 馬の距離適性取得

In [8]:
from tqdm import tqdm
import scraping
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup


DATA_DIR = Path("..", "data")
HTML_DIR = DATA_DIR / "html"
HTML_HORSE_DIR = HTML_DIR / "horse_java"

# DataFrameに格納する際の列名
columns = ["芝適正", "距離特性", "逃げ度", "早熟度", "重馬場適正"]

# 各horse_idについてDataFrameを作成
data = []
horse_ids = []
dfs = {}


html_paths_horse = list(scraping.HTML_HORSE_JAVA_DIR.glob("*.bin"))

html_paths_horse
for html_path in tqdm(html_paths_horse[:3]):
    horse_id = html_path.stem
    with open(html_path, "rb") as f:
        try:
            html = f.read()

            soup = BeautifulSoup(html, "html.parser")

            # すべてのimgタグを検索
            img_tags = soup.find_all("img")

            # `review_bar_blue.png` を含むimgタグを抽出
            target_imgs = [img for img in img_tags if any(keyword in img.get("src", "") for keyword in ["review_bar_blue.png", "review_bar_gray.png"])]

            width_list = []

            # 抽出したimgタグの属性を出力
            for img in target_imgs:
                # print("Found img tag:")
                # print("src:", img.get("src"))
                # print("width:", img.get("width"))
                # print("height:", img.get("height"))
                width = img.get("width")
                width_list.append(width)

            par = [width_list[i] for i in [0, 2, 4, 6, 8] if i < len(width_list)]
            data.append(par)
            horse_ids.append(horse_id)
        except:
            print(f"table not found at {horse_id}")
            continue
concat_df = pd.DataFrame(data, index=horse_ids, columns=columns)
concat_df.index.name = "horse_id"

concat_df

100%|██████████| 3/3 [00:02<00:00,  1.28it/s]


,芝適正,距離特性,逃げ度,早熟度,重馬場適正
horse_id,,,,,
2021100561,58,58,58,58,58
2017103876,58,58,58,58,58
2017104869,58,58,58,58,58


In [9]:
import create_rawdf

html_paths_horse = list(scraping.HTML_HORSE_JAVA_DIR.glob("*.bin"))
create_rawdf.create_horse_info(html_path_list=html_paths_horse[:10])

100%|██████████| 10/10 [00:08<00:00,  1.12it/s]


,horse_id,芝適正,距離特性,逃げ度,早熟度,重馬場適正
0,2021100561,58,58,58,58,58
1,2017103876,58,58,58,58,58
2,2017104869,58,58,58,58,58
3,2016106395,116,29,1,58,58
4,2016103888,58,58,58,58,58
5,2017100220,58,58,58,58,58
6,2015103392,116,116,116,116,19
7,2021104756,58,58,1,58,58
8,2017101073,58,58,58,58,58
9,2019100331,58,58,58,58,58


## 予測時の表作成

### 前日準備

# デバックスペース

In [1]:
from create_prediction_population import scrape_horse_id_list

scrape_horse_id_list(race_id='202506010306')

['2020101341',
 '2019106938',
 '2021104856',
 '2020102607',
 '2020104778',
 '2021101002',
 '2021103477',
 '2021100469',
 '2021102412',
 '2020102819',
 '2021100606',
 '2021103823',
 '2021104850']

In [6]:
html_path_list = html_paths_horse
for html_path in tqdm(html_path_list):
    with open(html_path, "rb") as f:
        
            horse_id = html_path.stem # Pathからhorse_idを抽出、bin部分を消している

            response = f.read()

            # エンコーディングを自動検出
            encoding = chardet.detect(response)['encoding']
            # print(f"検出されたエンコーディング: {encoding}")

            # 正しいエンコーディングでデコード
            html = response.decode(encoding)
            
            # 先ほどまでBeautifulSoupを使っていたのはhorse_idなどを抜くためである。ここでは必要ない
            
            # レースページのうち、3番目の表(レース結果)をdfに追加する
            # df = pd.read_html(html)[4]
            df = pd.read_html(StringIO(html))[4]

df

100%|██████████| 1/1 [00:00<00:00,  1.57it/s]


,0,1
0,コース適性,NaN
1,距離適性,NaN
2,脚質,NaN
3,成長,NaN
4,重馬場,NaN


In [3]:
horse_id_list = ["2022106681"]
html_paths_horse = scraping.scrape_html_horse(horse_id_list=horse_id_list, skip = False)
horse_results = create_rawdf.create_horse_results(
    html_path_list = html_paths_horse,
    save_filename="horse_results_prediction.csv"
    )

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  1.86it/s]


AttributeError: Can only use .str accessor with string values!